In [1]:
!hostname


c-11-01


In [2]:
import sys
import gzip
import glob
import numpy as np
import pandas as pd

import matplotlib
from matplotlib import pyplot as plt

In [3]:
def load_mgatk_output(output_dir, mito_length):
    # assuming mgatk output naming convention
    base_files = [glob.glob(output_dir + "*.{}.txt.gz".format(nt))[0] for nt in "ATCG"]

    base_coverage_dict = dict()
    for i, nt in enumerate("ATCG"):
        cur_base_data = pd.read_csv(gzip.open(base_files[i]), header=None)

        # gather coverage for forward strand
        fwd_base_df = cur_base_data[[0, 1, 2]].pivot_table(index=1, columns=0)
        fwd_base_df.columns = [
            x[1] for x in fwd_base_df.columns.values
        ]  # flatten weird multiindex after pivot
        fwd_base_df.index.name = None
        # missing_pos = [x for x in range(1, mito_length + 1) if x not in fwd_base_df.columns]
        # fwd_base_df[missing_pos] = 0  # fill in missing positions
        all_columns = [x for x in range(1, mito_length + 1)]
        fwd_base_df = fwd_base_df.reindex(columns=all_columns, fill_value=0)
        fwd_base_df = fwd_base_df.fillna(0).sort_index(
            axis=1
        )  # assume all nan are true zeroes

        # gather coverage for forward strand
        rev_base_df = cur_base_data[[0, 1, 3]].pivot_table(index=1, columns=0)
        rev_base_df.columns = [x[1] for x in rev_base_df.columns.values]
        rev_base_df.index.name = None
        # missing_pos = [x for x in range(1, mito_length + 1) if x not in rev_base_df.columns]
        # rev_base_df[missing_pos] = 0
        all_columns = [x for x in range(1, mito_length + 1)]
        rev_base_df = rev_base_df.reindex(columns=all_columns, fill_value=0)
        rev_base_df = rev_base_df.fillna(0).sort_index(axis=1)

        # organize base data into a dict
        base_coverage_dict[nt] = (fwd_base_df, rev_base_df)

    return base_coverage_dict


def gather_possible_variants(base_coverage_dict, reference_file):
    # sum across cells and strands for each base and position
    aggregated_genotype = pd.DataFrame(
        np.zeros((4, mito_length)),
        index=list("ATCG"),
        columns=np.arange(1, mito_length + 1),
    )
    for nt in base_coverage_dict:
        # sum across cells for each strand separately
        fwd_base_df, rev_base_df = base_coverage_dict[nt]
        fwd_base_sum, rev_base_sum = fwd_base_df.sum(), rev_base_df.sum()

        # sequencing artifact if a base/position is only nonzero for one strand across cells, ignore them
        masking = ~(
            (fwd_base_sum > 0) & (rev_base_sum > 0)
        )  # True if position not >0 for both strands
        fwd_base_sum[masking], rev_base_sum[masking] = 0, 0

        # sum across strands
        aggregated_genotype.loc[nt, :] = fwd_base_sum + rev_base_sum

    # make a reference set of tuples (pos, ref_base)
    ref_set = [x.strip().split() for x in open(reference_file, "r").readlines()]
    ref_N_positions = [int(x[0]) for x in ref_set if x[1].upper() not in letters]
    ref_set = set(
        [(int(x[0]), x[1].upper()) for x in ref_set if x[1].upper() in letters]
    )
    ref_dict = dict(ref_set)

    # make an observed set of tuples which are nonzero
    non_zero_idx = np.where(aggregated_genotype > 0)
    non_zero_bases = [letters[i] for i in non_zero_idx[0]]
    non_zero_pos = [int(i + 1) for i in non_zero_idx[1]]
    observed_set = list(zip(non_zero_pos, non_zero_bases))
    observed_set = set(
        [x for x in observed_set if x[0] not in ref_N_positions]
    )  # disregard positions in ref with N

    # take difference between observed and reference
    variant_set = observed_set - ref_set
    variants = sorted(
        [(x[0], ref_dict[x[0]], x[1]) for x in list(variant_set)], key=lambda x: x[0]
    )  # (pos, ref base, obs base)

    return variants


In [17]:
# sys.argv[1]
MGATK_OUT_DIR = "/scr1/users/liuc9/mitochondrial/realdata/01-Sci_Immunol_32651212/cromwell-executions/SCMTAH/022e0012-d8fc-4f0b-89d8-86004847b04a/call-call_mt_variants/execution/cell/final/"
sample_prefix = "cell"  # sys.argv[2]
mito_length = 16569  # int(sys.argv[3])  # 16569
low_coverage_threshold = 10  # int(sys.argv[4])  # 10
mito_genome = "MT"  # sys.argv[5]  # chrM


In [18]:
letters = list("ATCG")


In [19]:
# load_mgatk_output

output_dir = MGATK_OUT_DIR
mito_lenght = mito_length

base_files = [glob.glob(output_dir + "*.{}.txt.gz".format(nt))[0] for nt in "ATCG"]

base_coverage_dict = dict()

for i, nt in enumerate("ATCG"):
    cur_base_data = pd.read_csv(gzip.open(base_files[i]), header=None)


0
1
2
3


In [6]:
base_coverage_dict = load_mgatk_output(MGATK_OUT_DIR, mito_length)


IndexError: list index out of range

In [ ]:
cell_barcodes = base_coverage_dict["A"][0].index

In [6]:
cell_barcodes

Index(['cluster1-1', 'cluster2-1'], dtype='object')

In [36]:
total_coverage = pd.DataFrame(
    np.zeros((len(cell_barcodes), mito_length)),
    index=cell_barcodes,
    columns=np.arange(1, mito_length + 1),
)
for nt in base_coverage_dict:
    total_coverage += base_coverage_dict[nt][0]
    total_coverage += base_coverage_dict[nt][1]


In [37]:
cell_barcodes = total_coverage.index[
    total_coverage.mean(axis=1) > low_coverage_threshold
]
for nt in base_coverage_dict:
    base_coverage_dict[nt] = (
        base_coverage_dict[nt][0].loc[cell_barcodes, :],
        base_coverage_dict[nt][1].loc[cell_barcodes, :],
    )
total_coverage = total_coverage.loc[cell_barcodes, :]


In [38]:
variants = gather_possible_variants(
    base_coverage_dict, MGATK_OUT_DIR + mito_genome + "_refAllele.txt"
)
variant_names = ["{}{}>{}".format(x[0], x[1], x[2]) for x in variants]


In [40]:
# build two <cell by variant tables>, one for each strand
total_coverage_variant_df = []
fwd_cell_variant_df, rev_cell_variant_df = [], []

In [41]:
for i, var in enumerate(variants):
    var_name = variant_names[i]
    pos, base = var[0], var[2]
    total_coverage_variant_df.append(total_coverage[pos])
    fwd_cell_variant_df.append(base_coverage_dict[base][0][pos].values)
    rev_cell_variant_df.append(base_coverage_dict[base][1][pos].values)

In [43]:
total_coverage_variant_df

[]

In [42]:
total_coverage_variant_df = pd.DataFrame(
    np.array(total_coverage_variant_df).T, index=cell_barcodes, columns=variant_names
)


ValueError: Empty data passed with indices specified.